In [2]:
# ═══════════════════════════════════════════════
# DR Congo PEP → 제재 대조 → 제재 인물 뉴스 모니터
# ═══════════════════════════════════════════════
import sys, subprocess, importlib
try:
    import feedparser
except ImportError:                                   # 없으면 현재 환경에 자동 설치
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "feedparser"])
    importlib.invalidate_caches()
    import feedparser

import os, re, time, unicodedata
from collections import defaultdict
from datetime import datetime
from urllib.parse import urljoin, quote
import requests
import pandas as pd
from bs4 import BeautifulSoup

BASE = "https://www.opensanctions.org"
HEADERS = {"User-Agent": "Mozilla/5.0 (study project)"}
COUNTRY = "DR Congo"
PREFIX = "drc"
EXPECTED = 561

LOOP = False                  # True: 30분마다 자동 업데이트 / False: 한 번만 실행
INTERVAL = 30 * 60
WATCH_EXTRA = []              # 감시에 직접 추가할 이름 (예: ["Felix Tshisekedi"])

# 제재 카테고리 판단용 기관 가정 (한국 금융기관 기준 예시 — 실제 기관 상황에 맞게 수정)
INSTITUTION = {"eu_nexus": False,   # EU 지점·EU 거래 연결고리가 있는가
               "uk_nexus": False,   # 영국 연결고리
               "usd_nexus": True}   # 달러 결제가 있는가 (OFAC 영향)


def strip_accents(s):
    return "".join(c for c in unicodedata.normalize("NFKD", str(s)) if not unicodedata.combining(c))


def tokens(s):
    return [t for t in re.split(r"[\s\-,.']+", strip_accents(s).lower()) if len(t) > 1]


# ───────────────────────────────────────────────
# 1단계: /pep/ 에서 DR Congo 데이터셋 찾기 → CSV 받기
# ───────────────────────────────────────────────
res = requests.get(f"{BASE}/pep/", headers=HEADERS, timeout=10)
print("상태 코드:", res.status_code)
if res.status_code != 200 or COUNTRY not in res.text:
    raise SystemExit("페이지를 못 받았거나 국가 이름이 없어요.")

soup = BeautifulSoup(res.text, "html.parser")
sources = {}
for a in soup.find_all("a", href=True):
    href, text = a["href"], a.get_text(strip=True)
    if href.startswith("/datasets/") and COUNTRY in text:
        sources[href.strip("/").split("/")[-1]] = text

print(f"\n{COUNTRY} 데이터셋 {len(sources)}개")
for slug, name in sources.items():
    print(f" - {name} ({slug})")
if not sources:
    raise SystemExit("데이터셋 링크를 못 찾았어요.")

frames = []
for slug, name in sources.items():
    time.sleep(10)                                   # robots.txt Crawl-delay 준수
    ds_url = f"{BASE}/datasets/{slug}/"
    ds_soup = BeautifulSoup(requests.get(ds_url, headers=HEADERS, timeout=10).text, "html.parser")
    links = [urljoin(ds_url, a["href"]) for a in ds_soup.find_all("a", href=True) if ".csv" in a["href"]]
    csv_url = links[0] if links else f"https://data.opensanctions.org/datasets/latest/{slug}/targets.simple.csv"
    df_src = pd.read_csv(csv_url)
    df_src["source"] = name
    frames.append(df_src)
    print(f"✅ {name}: {len(df_src)}행")

peps = pd.concat(frames, ignore_index=True)
if "schema" in peps:
    peps = peps[peps["schema"] == "Person"]
peps = peps.drop_duplicates("id").reset_index(drop=True)
print(f"\n인물 {len(peps)}명 (예상 약 {EXPECTED}명)")

# ───────────────────────────────────────────────
# 2단계: 제재명단 대조 (ID 확정 + 어순 무관 이름·별칭 후보)
# ───────────────────────────────────────────────
SANC_URL = "https://data.opensanctions.org/datasets/latest/sanctions/targets.simple.csv"
SANC_FILE = "sanctions_cache_v2.csv"
keep = {"id", "schema", "name", "aliases", "birth_date", "countries", "sanctions", "dataset"}

if os.path.exists(SANC_FILE):
    sanc = pd.read_csv(SANC_FILE, low_memory=False)
else:
    print("\n제재명단 다운로드 중... (처음 한 번만)")
    sanc = pd.read_csv(SANC_URL, usecols=lambda c: c in keep, low_memory=False)
    sanc.to_csv(SANC_FILE, index=False)
print(f"제재명단 {len(sanc):,}건")

id_hits = peps.merge(sanc, on="id", how="inner", suffixes=("", "_sanc"))

index = defaultdict(list)
for r in sanc[sanc["schema"] == "Person"].to_dict("records"):
    names = [r["name"]] + (str(r["aliases"]).split(";") if pd.notna(r.get("aliases")) else [])
    for n in names:
        tk = frozenset(tokens(n))
        if len(tk) >= 2:
            for t in tk:
                index[t].append((tk, r["id"], n, r.get("birth_date"), r.get("dataset"), r.get("sanctions")))

rows = []
for p in peps.to_dict("records"):
    ptk, seen_ids = set(tokens(p["name"])), set()
    pbd = str(p.get("birth_date"))[:10]
    for t in ptk:
        for tk, sid, sname, sbd, sds, sprog in index.get(t, []):
            if sid in seen_ids or not tk <= ptk:
                continue
            seen_ids.add(sid)
            rows.append({"pep_id": p["id"], "pep_name": p["name"], "pep_birth": p.get("birth_date"),
                         "sanc_id": sid, "matched_name": sname, "sanc_birth": sbd,
                         "birth_match": bool(pd.notna(sbd) and pbd != "nan" and str(sbd)[:10] == pbd),
                         "sanc_dataset": sds, "sanc_programs": sprog})

cands = pd.DataFrame(rows, columns=["pep_id", "pep_name", "pep_birth", "sanc_id",
                                    "matched_name", "sanc_birth", "birth_match", "sanc_dataset",
                                    "sanc_programs"])
cands = cands[~cands["pep_id"].isin(id_hits["id"])].copy()
cands["verdict"] = cands["birth_match"].map(
    {True: "동일인 가능성 높음 (생년월일 일치)", False: "오탐 가능성 (생년월일 불일치·없음)"})

print(f"\n[제재 ID 일치 — 확정] {len(id_hits)}명")
if len(id_hits):
    cols = [c for c in ["name", "birth_date", "sanctions", "dataset_sanc"] if c in id_hits.columns]
    print(id_hits[cols].to_string(index=False))
print(f"\n[이름·별칭 일치 — 후보] {len(cands)}건 (생년월일 일치 {int(cands['birth_match'].sum())}건)")
if len(cands):
    print(cands[["pep_name", "matched_name", "verdict"]].to_string(index=False))

# ───────────────────────────────────────────────
# 3단계: 감시 대상 정하기 (확정 → 생년월일 일치 후보 → 후보 전체)
# ───────────────────────────────────────────────
watch = list(id_hits["name"]) + cands.loc[cands["birth_match"], "pep_name"].tolist()
if not watch:
    watch = cands["pep_name"].tolist()
watch = list(dict.fromkeys(watch + WATCH_EXTRA))

print(f"\n감시 대상 {len(watch)}명")
for n in watch:
    print(" -", n)
if not watch:
    raise SystemExit("감시할 제재 인물이 없어요. WATCH_EXTRA에 이름을 직접 넣어보세요.")


# ───────────────────────────────────────────────
# 4단계: 인물 이름으로 구글 뉴스 RSS 검색 (프랑스어·콩고판)
# ───────────────────────────────────────────────
def short(n):                                        # 후성·부칭 빼고 이름+성
    t = n.split()
    return f"{t[0]} {t[-1]}" if len(t) >= 3 else n


def query_for(n):
    # 콩고 공식 명단은 "성 후성 이름" 순서, 기사는 보통 "이름 성" 순서 → 두 순서 모두 검색
    t = n.split()
    a, b = (t[0], t[-1]) if len(t) >= 2 else (n, "")
    return f'"{b} {a}" OR "{a} {b}"' if b else f'"{a}"'


FEED_PERSON = {}
for n in watch:
    q = quote(query_for(n))
    FEED_PERSON[f"https://news.google.com/rss/search?q={q}&hl=fr&gl=CD&ceid=CD:fr"] = n


def fetch_person_news():
    items = []
    for url, person in FEED_PERSON.items():
        # requests로 받아서(리다이렉트 자동 추적) feedparser에 넘김
        try:
            r = requests.get(url, headers=HEADERS, timeout=20, allow_redirects=True)
        except Exception as e:
            print(f"  ⚠ {person}: 요청 실패 ({e})")
            continue
        feed = feedparser.parse(r.content)
        if r.status_code != 200 or not feed.entries:
            print(f"  ⚠ {person}: 상태 {r.status_code}, 기사 {len(feed.entries)}건, 최종 주소 {r.url[:80]}")
            if "consent" in r.url:
                print("     → 구글 동의(consent) 페이지로 이동됨: 브라우저에서 위 주소를 한 번 열어보세요")
        print(f"  {person}: {len(feed.entries)}건")
        for e in feed.entries:
            items.append({"person": person,
                          "title": e.get("title"),
                          "source": e.get("source", {}).get("title", ""),
                          "published": e.get("published", "")[:25],
                          "url": e.get("link")})
        time.sleep(3)                                # 인물별 요청 간격
    return items


# ───────────────────────────────────────────────
# 5단계: 새 기사만 표시·저장 (LOOP=True면 자동 반복)
# ───────────────────────────────────────────────
OUT = f"{PREFIX}_sanction_news_monitor.csv"
seen = set(pd.read_csv(OUT)["url"]) if os.path.exists(OUT) else set()
first_run = not seen

while True:
    print(f"\n[{datetime.now():%Y-%m-%d %H:%M}] {len(watch)}명 뉴스 확인 중...")
    articles = fetch_person_news()
    new = [a for a in articles if a["url"] and a["url"] not in seen]
    for a in new:
        seen.add(a["url"])
        a["found_at"] = f"{datetime.now():%Y-%m-%d %H:%M}"

    label = "기존 기사(첫 실행)" if first_run else "새 기사"
    print(f"  받은 기사 {len(articles)}건 · {label} {len(new)}건")
    for a in sorted(new, key=lambda x: x["person"])[:30]:
        print(f"  🔔 {a['person']} | {a['title']} ({a['source']}, {a['published'][:16]})")

    if new:
        pd.DataFrame(new).to_csv(OUT, mode="a", index=False,
                                 header=not os.path.exists(OUT), encoding="utf-8-sig")
    first_run = False
    if not LOOP:
        break
    time.sleep(INTERVAL)

# ───────────────────────────────────────────────
# 6단계: 결과 저장 (분석용, 재배포 금지)
# ───────────────────────────────────────────────
peps.to_csv(f"{PREFIX}_peps.csv", index=False, encoding="utf-8-sig")
id_hits.to_csv(f"{PREFIX}_sanction_id_hits.csv", index=False, encoding="utf-8-sig")
cands.to_csv(f"{PREFIX}_sanction_candidates.csv", index=False, encoding="utf-8-sig")

# ───────────────────────────────────────────────
# 7단계: 리포트용 요약
# ───────────────────────────────────────────────
def n_lists(x):
    return len(str(x).split(";")) if pd.notna(x) else 0

summary = []
for _, r in id_hits.iterrows():
    summary.append({"인물": r["name"], "판정": "확정 (ID 일치)",
                    "제재 목록 수": n_lists(r.get("dataset_sanc")), "제재 목록": r.get("dataset_sanc")})
for _, r in cands.iterrows():
    summary.append({"인물": r["pep_name"], "판정": r["verdict"],
                    "제재 목록 수": n_lists(r["sanc_dataset"]), "제재 목록": r["sanc_dataset"]})
summary = pd.DataFrame(summary)

news_all = pd.read_csv(OUT) if os.path.exists(OUT) else pd.DataFrame(columns=["person"])
news_cnt = news_all["person"].value_counts()
if len(summary):
    summary["누적 기사 수"] = summary["인물"].map(news_cnt).fillna(0).astype(int)
    summary.to_csv(f"{PREFIX}_summary.csv", index=False, encoding="utf-8-sig")

print("\n" + "═" * 60)
print(f"요약: {COUNTRY} 의원 {len(peps)}명 중")
print(f"  · 제재 확정 {len(id_hits)}명")
print(f"  · 동일인 가능성 높은 후보 {int(cands['birth_match'].sum())}명")
print(f"  · 오탐으로 판단한 후보 {int((~cands['birth_match']).sum())}명")
if len(summary):
    print(summary[["인물", "판정", "제재 목록 수", "누적 기사 수"]].to_string(index=False))
print("═" * 60)
print("\n✅ 저장 완료")

# ───────────────────────────────────────────────
# 7-2단계: 제재 출처·명목·카테고리(A 금지 / B 제한 / C 감시) 라벨링
# ───────────────────────────────────────────────
from IPython.display import display
pd.set_option("display.max_colwidth", None)

# 목록 이름 → (원지정 기관, 독자/이행, 조치 종류)
LIST_RULES = [
    ("un security council",    "UN 안보리",   "원지정", "자산동결·여행금지"),
    ("ofac",                   "미국 OFAC",   "원지정", "자산동결"),
    ("eu consolidated travel", "EU",          "원지정", "여행금지"),
    ("eu council official",    "EU",          "원지정", "법적 지정 근거(관보)"),
    ("eu financial sanctions", "EU",          "원지정", "자산동결"),
    ("uk fcdo",                "영국",        "원지정", "자산동결"),
    ("belgian",                "EU",          "이행(벨기에)", "자산동결"),
    ("french",                 "EU",          "이행(프랑스)", "자산동결"),
    ("monaco",                 "EU",          "이행(모나코)", "자산동결"),
    ("swiss",                  "EU",          "이행(스위스)", "자산동결"),
    ("australian",             "UN 안보리",   "이행(호주)", "자산동결"),
    ("japan",                  "UN 안보리",   "이행(일본)", "자산동결"),
    ("south africa",           "UN 안보리",   "이행(남아공)", "자산동결"),
    ("argentina",              "UN 안보리",   "이행(아르헨티나)", "자산동결"),
    ("strategic high-tech",    "대만",        "독자", "수출통제"),
    ("sfms blacklist",         "우크라이나",  "독자", "FIU 감시목록"),
    ("iran",                   "이란",        "독자", "자산동결"),
]

def label_list(lst):
    s_ = " " + lst.lower() + " "
    for key, origin, kind, measure in LIST_RULES:
        if key in s_:
            return origin, kind, measure
    return "기타", "독자", "확인 필요"

def category_of(origin, measure):
    """우리 기관(INSTITUTION 가정)에 대한 구속력과 조치 종류로 A/B/C 판정"""
    freeze = "자산동결" in measure
    binding = (origin == "UN 안보리"
               or (origin == "미국 OFAC" and INSTITUTION["usd_nexus"])
               or (origin == "EU" and INSTITUTION["eu_nexus"])
               or (origin == "영국" and INSTITUTION["uk_nexus"]))
    if binding and freeze:
        return "A", "구속력 있는 자산동결 → 거래 금지·동결·보고 (점수 계산 없이 오버라이드)"
    if freeze or measure in ("수출통제",):
        return "B", "구속력 없는 자산동결 또는 수출통제 → EDD·상급자 승인·거래 제한"
    return "C", "여행금지·감시목록 등 → 위험 요인으로 반영, 모니터링 강화"

def program_of(text):
    """OpenSanctions 'sanctions' 필드에서 제재 명목(프로그램) 요약"""
    if pd.isna(text) or not str(text).strip():
        return "명목 정보 없음 (원 공시 확인 필요)"
    t = str(text)
    tags = []
    for key, tag in [("congo", "DR Congo 제재 프로그램"), ("terror", "테러"),
                     ("iran", "이란 제재"), ("human rights", "인권 침해"),
                     ("arms", "무기 금수 위반"), ("armed group", "무장단체"),
                     ("corruption", "부패")]:
        if key in t.lower() and tag not in tags:
            tags.append(tag)
    head = t.replace(";", " / ")[:160]
    return (", ".join(tags) + " | " if tags else "") + head

# 대상: 확정 + 동일인 후보 (오탐 제외)
targets = []
prog_col = "sanctions_sanc" if "sanctions_sanc" in id_hits.columns else "sanctions"
for _, r in id_hits.iterrows():
    targets.append((r["name"], "확정", r.get("dataset_sanc"), r.get(prog_col)))
for _, r in cands[cands["birth_match"]].iterrows():
    targets.append((r["pep_name"], "동일인 후보", r["sanc_dataset"], r.get("sanc_programs")))

RANK = {"A": 0, "B": 1, "C": 2}
lab_rows = []
for person, verdict, lists, progs in targets:
    for lst in str(lists).split(";"):
        lst = lst.strip()
        if not lst or lst == "nan":
            continue
        origin, kind, measure = label_list(lst)
        cat, why = category_of(origin, measure)
        lab_rows.append({"인물": person, "판정": verdict, "제재 목록": lst,
                         "원지정 기관": origin, "구분": kind, "조치": measure,
                         "카테고리": cat, "근거": why})
labeled = pd.DataFrame(lab_rows)

cat_rows = []
if len(labeled):
    labeled["_r"] = labeled["카테고리"].map(RANK)
    labeled = labeled.sort_values(["인물", "_r"]).drop(columns="_r")

    print("\n" + "═" * 60)
    print("제재 출처·명목·카테고리")
    print(f"기관 가정: EU 연결 {INSTITUTION['eu_nexus']}, 영국 연결 {INSTITUTION['uk_nexus']}, "
          f"달러 결제 {INSTITUTION['usd_nexus']}")
    print("═" * 60)
    display(labeled[["인물", "제재 목록", "원지정 기관", "구분", "조치", "카테고리"]].reset_index(drop=True))

    for person, verdict, lists, progs in targets:
        g = labeled[labeled["인물"] == person]
        if g.empty:
            continue
        top = g.iloc[0]
        origins = sorted(set(g["원지정 기관"]) - {"대만", "우크라이나"})
        cat_rows.append({"인물": person, "판정": verdict,
                         "최종 카테고리": top["카테고리"],
                         "가장 강한 제재": f"{top['원지정 기관']} · {top['조치']}",
                         "독립 원지정 기관": ", ".join(origins),
                         "목록 수": len(g),
                         "제재 명목": program_of(progs),
                         "처리 방식": top["근거"]})
    cat_df = pd.DataFrame(cat_rows).sort_values("최종 카테고리")
    print("\n[인물별 최종 카테고리]")
    display(cat_df.reset_index(drop=True))
    for _, r in cat_df.iterrows():
        print(f"\n■ {r['인물']} → {r['최종 카테고리']}급")
        print(f"  가장 강한 제재: {r['가장 강한 제재']} (독립 원지정: {r['독립 원지정 기관']}, 목록 {r['목록 수']}개)")
        print(f"  명목: {r['제재 명목']}")
        print(f"  처리: {r['처리 방식']}")
    labeled.to_csv(f"{PREFIX}_sanction_labeled.csv", index=False, encoding="utf-8-sig")
    cat_df.to_csv(f"{PREFIX}_sanction_category.csv", index=False, encoding="utf-8-sig")
    print(f"\n✅ 저장: {PREFIX}_sanction_labeled.csv, {PREFIX}_sanction_category.csv")
    print("   ※ 카테고리 기준과 기관 가정은 설명용 예시이며, 실제 적용은 기관 정책·법적 검토가 필요해요.")

# ───────────────────────────────────────────────
# 8단계: 인물별 최신 기사 5건 — 원문 헤드라인 + 한글 번역
# ───────────────────────────────────────────────
try:
    import deep_translator
except ImportError:
    print("\n번역 라이브러리(deep-translator) 설치 중...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "deep-translator"])
    importlib.invalidate_caches()
from deep_translator import GoogleTranslator, MyMemoryTranslator
from IPython.display import display

pd.set_option("display.max_colwidth", None)

# 번역 엔진 선택: 구글 → 안 되면 MyMemory → 둘 다 안 되면 번역 생략
TEST = "Bonjour, comment allez-vous ?"
ENGINES = [("Google", lambda: GoogleTranslator(source="auto", target="ko")),
           ("MyMemory", lambda: MyMemoryTranslator(source="fr-FR", target="ko-KR"))]
translator, engine_name = None, None
for name, make in ENGINES:
    try:
        t = make()
        out = t.translate(TEST)
        if out and out.strip() != TEST:
            translator, engine_name = t, name
            print(f"\n번역 엔진: {name} (테스트: '{TEST}' → '{out}')")
            break
    except Exception as e:
        print(f"\n⚠ {name} 번역 실패: {type(e).__name__} {str(e)[:100]}")
if translator is None:
    print("⚠ 사용 가능한 번역 엔진이 없어 번역 없이 원문만 표시해요.")


def to_korean(text):
    if translator is None:
        return "(번역 불가)"
    try:
        result = translator.translate(str(text))
        time.sleep(0.5)                               # 번역 요청 간격
        return result
    except Exception as e:
        return f"(번역 실패: {type(e).__name__})"


def clean_headline(title, source):
    # 구글 뉴스 제목 끝의 " - 언론사명" 제거
    t, s = str(title), str(source)
    return t.rsplit(" - ", 1)[0] if s and t.endswith(s) else t


news = pd.read_csv(OUT)
news["date"] = pd.to_datetime(news["published"], errors="coerce", utc=True).dt.strftime("%Y-%m-%d")
news["headline"] = [clean_headline(t, s) for t, s in zip(news["title"], news["source"].fillna(""))]

top_all = []
for person in watch:
    top = (news[news["person"] == person]
           .sort_values("date", ascending=False)
           .drop_duplicates("headline")
           .head(5)
           .copy())
    top["한글 번역"] = [to_korean(h) for h in top["headline"]]
    top_all.append(top)
    print(f"\n■ {person} — 최신 기사 {len(top)}건")
    display(top[["date", "source", "headline", "한글 번역"]]
            .rename(columns={"date": "날짜", "source": "언론사", "headline": "원문 헤드라인"})
            .reset_index(drop=True))

if top_all:
    result = pd.concat(top_all)
    result.to_csv(f"{PREFIX}_news_top5_ko.csv", index=False, encoding="utf-8-sig")
    ok = (~result["한글 번역"].str.startswith("(")).sum()
    print(f"\n✅ 번역본 저장: {PREFIX}_news_top5_ko.csv · 번역 성공 {ok}/{len(result)}건 (엔진: {engine_name})")
    print("   기계 번역이라 인명·정치 용어는 원문과 함께 확인하세요.")

상태 코드: 200

DR Congo 데이터셋 1개
 - DR Congo Members of the National Assembly (cd_national_assembly)
✅ DR Congo Members of the National Assembly: 560행

인물 560명 (예상 약 561명)
제재명단 73,810건

[제재 ID 일치 — 확정] 1명
                        name birth_date  sanctions                                                                                                                                                                                                                                       dataset_sanc
Bitakwira Bihona Hayi Justin 1960-12-05        NaN Belgian Financial Sanctions;EU Consolidated Travel Bans;EU Council Official Journal Sanctioned Entities;EU Financial Sanctions Files (FSF);French National Asset Freezing System;Monaco National Fund Freezing List;Swiss SECO Sanctions/Embargoes

[이름·별칭 일치 — 후보] 2건 (생년월일 일치 1건)
                        pep_name     matched_name              verdict
          Ndjabu Ngabu Floribert Floribert Ndjabu 동일인 가능성 높음 (생년월일 일치)
Wagner Tshienda Tshienda Richard   R

,인물,제재 목록,원지정 기관,구분,조치,카테고리
0,Bitakwira Bihona Hayi Justin,Belgian Financial Sanctions,EU,이행(벨기에),자산동결,B
1,Bitakwira Bihona Hayi Justin,EU Financial Sanctions Files (FSF),EU,원지정,자산동결,B
2,Bitakwira Bihona Hayi Justin,French National Asset Freezing System,EU,이행(프랑스),자산동결,B
3,Bitakwira Bihona Hayi Justin,Monaco National Fund Freezing List,EU,이행(모나코),자산동결,B
4,Bitakwira Bihona Hayi Justin,Swiss SECO Sanctions/Embargoes,EU,이행(스위스),자산동결,B
5,Bitakwira Bihona Hayi Justin,EU Consolidated Travel Bans,EU,원지정,여행금지,C
6,Bitakwira Bihona Hayi Justin,EU Council Official Journal Sanctioned Entities,EU,원지정,법적 지정 근거(관보),C
7,Ndjabu Ngabu Floribert,Argentina RePET Sanctions,UN 안보리,이행(아르헨티나),자산동결,A
8,Ndjabu Ngabu Floribert,Australian Sanctions Consolidated List,UN 안보리,이행(호주),자산동결,A
9,Ndjabu Ngabu Floribert,Japan Economic Sanctions and List of Eligible People,UN 안보리,이행(일본),자산동결,A



[인물별 최종 카테고리]


,인물,판정,최종 카테고리,가장 강한 제재,독립 원지정 기관,목록 수,제재 명목,처리 방식
0,Ndjabu Ngabu Floribert,동일인 후보,A,UN 안보리 · 자산동결,"EU, UN 안보리, 영국",15,"DR Congo 제재 프로그램, 인권 침해, 무기 금수 위반, 무장단체 | (UE) 2017/396 du 07/03/2017 (ONU République démocratique du Congo - RCSNU 1533 (2004) et R (CE) 1183/2005) - décision du comité des sanctions des Nations unies",구속력 있는 자산동결 → 거래 금지·동결·보고 (점수 계산 없이 오버라이드)
1,Bitakwira Bihona Hayi Justin,확정,B,EU · 자산동결,EU,7,"DR Congo 제재 프로그램, 무장단체 | (UE) 2025/2508 du 08/12/2025 (UE République démocratique du Congo - R (UE) 1183/2005) - (UE) 2022/2397 du 08/12/2022 (UE République démocratique du Congo - R (U",구속력 없는 자산동결 또는 수출통제 → EDD·상급자 승인·거래 제한



■ Ndjabu Ngabu Floribert → A급
  가장 강한 제재: UN 안보리 · 자산동결 (독립 원지정: EU, UN 안보리, 영국, 목록 15개)
  명목: DR Congo 제재 프로그램, 인권 침해, 무기 금수 위반, 무장단체 | (UE) 2017/396 du 07/03/2017 (ONU République démocratique du Congo - RCSNU 1533 (2004) et R (CE) 1183/2005) - décision du comité des sanctions des Nations unies 
  처리: 구속력 있는 자산동결 → 거래 금지·동결·보고 (점수 계산 없이 오버라이드)

■ Bitakwira Bihona Hayi Justin → B급
  가장 강한 제재: EU · 자산동결 (독립 원지정: EU, 목록 7개)
  명목: DR Congo 제재 프로그램, 무장단체 | (UE) 2025/2508 du 08/12/2025 (UE République démocratique du Congo - R (UE) 1183/2005) - (UE) 2022/2397 du 08/12/2022 (UE République démocratique du Congo - R (U
  처리: 구속력 없는 자산동결 또는 수출통제 → EDD·상급자 승인·거래 제한

✅ 저장: drc_sanction_labeled.csv, drc_sanction_category.csv
   ※ 카테고리 기준과 기관 가정은 설명용 예시이며, 실제 적용은 기관 정책·법적 검토가 필요해요.

⚠ Google 번역 실패: TooManyRequests Server Error: You made too many requests to the server.According to google, you are allowed to make 

번역 엔진: MyMemory (테스트: 'Bonjour, comment allez-vous ?' → '안녕하세요, 잘 지내시죠?')

■ Bitakwir

,날짜,언론사,원문 헤드라인,한글 번역
0,2026-07-11,mediacongo.net,Actualité | Sud-Kivu : L’élu d’Uvira Justin Bitakwira tire la sonnette d’alarme sur l’avancée des rebelles vers Baraka,뉴스 | 남부 키부: 우비라의 선출직 대표 저스틴 비타키라 (Justin Bitakwira) 는 바라카를 향한 반군의 진격에 대해 경종을 울린다.
1,2026-06-08,mediacongo.net,Actualité | Justin Bitakwira plaide pour un dialogue entre majorité et opposition afin d’éviter une crise politique,"뉴스 | 저스틴 비타크위라 (Justin Bitakwira), 정치적 위기를 피하기 위해 다수당과 야당 간의 대화 촉구"
2,2026-04-30,mediacongo.net,Actualité | Assemblée nationale : Justin Bitakwira appelle les parlementaires à se concentrer sur la guerre plutôt que sur les querelles politiques,"뉴스 | 국회: 저스틴 비타크위라 (Justin Bitakwira), 국회의원들에게 정치적 다툼보다는 전쟁에 집중할 것을 촉구"
3,2026-03-19,ACP | Agence Congolaise de Presse,Faux ! Justin Bitakwira n’est pas contre le changement de la Constitution,틀렸습니다! 저스틴 비타크위라는 헌법을 바꾸는 것을 반대하지 않습니다
4,2025-12-30,Pressenza - International Press Agency,[RD Congo] Rumonge : Justin Bitakwira apporte une aide d’urgence à plus de 1000 réfugiés congolais en détresse - Pressenza,"[DR 콩고] 루몬지: 저스틴 비타크위라, 1,000명 이상의 콩고 난민에게 긴급 지원 제공 - 프레센자"



■ Ndjabu Ngabu Floribert — 최신 기사 5건


,날짜,언론사,원문 헤드라인,한글 번역
0,2023-02-03,DW.com,"Floribert Ndjabu : ""Ils acceptent de déposer les armes""","플로리베르트 은자부: ""그들은 팔을 내려놓는 것에 동의합니다"""
1,2023-01-25,DW.com,"""La situation doit d'abord se calmer"" - Floribert Ndjabu Ngabu","""상황이 먼저 진정되어야 합니다"" - Floribert Ndjabu Ngabu"
2,2022-04-12,RFI,RDC: Thomas Lubanga et Floribert Ndjabu retrouvent la liberté en Ituri,DRC: 토마스 루방가 (Thomas Lubanga) 와 플로리버트 은자부 (Floribert Ndjabu) 가 이투리에서 자유를
3,2022-04-12,Actualite.cd,"RDC: Thomas Lubanga, Floribert Ndjabu et deux officiers de l’armée finalement libérés par les miliciens CODECO","DRC: 토마스 루방가, 플로리베르트 은자부, 그리고 마침내 코데코 민병대에 의해 석방된 두 명의 육군 장교"
4,2022-03-11,Actualite.cd,"RDC: Thomas Lubanga, Germain Katanga, Floribert Ndjabu et 5 autres membres de leur délégation toujours en otage","DRC: Thomas Lubanga, Germain Katanga, Floribert Ndjabu 및 다른 대표단 5 명은 여전히 인질로 잡혀 있습니다."



✅ 번역본 저장: drc_news_top5_ko.csv · 번역 성공 10/10건 (엔진: MyMemory)
   기계 번역이라 인명·정치 용어는 원문과 함께 확인하세요.


# DR Congo PEP 제재 스크리닝 및 뉴스 헤드라인 수집 EDA 리포트

## 1. 목표 질문

특정 국가(콩고민주공화국, DR Congo)의 PEP 명단을 발췌한 뒤, 두 가지 질문에 답하는 것을 목표로 했다. 첫째, 명단의 인물 중 **제재 대상**이 있는가. 둘째, 제재 대상 인물에 관한 **뉴스 헤드라인**을 자동으로 찾을 수 있는가. 이는 금융기관 KYC 절차의 제재 스크리닝과 부정적 언론 검색(adverse media screening)을 축소해 재현한 것이다.

여기에 더해, 제재 대상이 확인되었을 때 **어디서 어떤 제재를 받았는지 라벨링하고, 그 제재가 금융거래에 얼마나 치명적인지 등급(A/B/C)과 치명도로 정량화**하는 단계까지 확장했다.

이번 분석은 앞선 알바니아 분석의 후속이다. 알바니아에서는 제재 대조와 뉴스 검색이 모두 0건으로 나왔고, 원인을 추적한 결과 결과 자체보다 **이름 매칭 방식과 수집 구조의 문제**였음을 확인했다. DR Congo 분석은 그 교훈을 반영해 파이프라인을 개선하고, 개선된 방식이 실제로 의미 있는 결과를 내는지 검증하는 데 목적이 있다.

## 2. 데이터와 방법

### 2.1 대상 국가 선정

DR Congo를 선택한 이유는 두 가지다. 첫째, OpenSanctions `/pep/` 표 기준 국회의원이 561명으로 알바니아(141명)보다 표본이 크다. 둘째, DR Congo는 UN 안보리 제재 체제가 장기간 운영되어 온 국가로, 제재 대조에서 실제 일치 사례가 나올 가능성이 높아 **개선된 매칭 방식을 검증하기에 적합**하다고 판단했다.

### 2.2 명단 수집

PEP 명단은 OpenSanctions의 `/pep/` 페이지에서 BeautifulSoup으로 "DR Congo"가 포함된 데이터셋 링크를 찾고, 해당 데이터셋의 공식 CSV를 내려받았다. 알바니아 분석에서 이 페이지가 HTML 소스에 데이터가 들어 있는 정적(서버 렌더링) 페이지임을 확인했으므로 `requests` + BeautifulSoup을 그대로 사용했다. robots.txt의 Crawl-delay에 따라 요청 사이에 10초 간격을 두었다.

### 2.3 제재 대조: 알바니아에서 개선한 점

제재 여부는 OpenSanctions의 **제재명단 통합 파일**(73,810건)과 대조했다. 알바니아에서는 이름을 글자 그대로 비교하는 정확 일치 방식을 썼는데, 공식 명단에 부칭(아버지 이름)이 끼어 있어 매칭이 실패했다. DR Congo의 이름도 "성 + 후성(postnom) + 이름" 순서처럼 일반적인 기사·제재명단 표기와 어순이 달라 같은 문제가 반복될 것으로 예상했다. 이에 대조 방식을 세 단계로 바꿨다.

| 단계 | 방법 | 판정 |
|---|---|---|
| 1 | OpenSanctions가 부여한 공통 ID 일치 | 확정 |
| 2 | 제재명단 이름·별칭(aliases)의 단어 묶음이 PEP 이름에 모두 포함 (어순 무관) | 후보 |
| 3 | 후보의 생년월일 비교 | 일치 시 "동일인 가능성 높음", 불일치 시 "오탐" |

2단계는 넓게 잡는 대신 흔한 이름끼리 우연히 겹칠 위험이 있어, 3단계 생년월일 대조로 동명이인을 걸러내도록 설계했다.

### 2.4 제재 라벨링과 카테고리 분류

제재 대상이 확인되면 "제재 대상이다"에서 멈추지 않고, 각 제재 목록에 대해 **원지정 기관**(UN, EU, 영국 등), **원지정인지 타국 이행인지**, **조치 종류**(자산동결, 여행금지, 수출통제 등)를 라벨링했다. 각국 목록의 상당수는 UN이나 EU의 결정을 국내법으로 이행한 것이어서, 목록 수를 그대로 세면 같은 결정을 여러 번 세는 문제가 생기기 때문이다.

그다음 각 제재를 **우리 기관에 대한 구속력과 조치 종류**에 따라 세 카테고리로 분류했다. 기관은 "한국 금융기관, EU·영국 연결 없음, 달러 결제 있음"으로 가정했다.

| 카테고리 | 기준 | 처리 방식 |
|---|---|---|
| A 금지 | 우리 기관에 구속력 있는 자산동결 (UN 안보리, 달러 결제 시 OFAC 등) | 점수 계산 없이 거래 금지·동결·보고 (오버라이드) |
| B 제한 | 구속력 없는 자산동결, 수출통제 | EDD, 상급자 승인, 고위험 하한 적용 |
| C 감시 | 여행금지, 감시목록, 지정 근거 공시 등 | 위험 요인으로 반영, 모니터링 강화 |

A급을 점수에 합산하지 않고 오버라이드로 처리한 것은 다기준 의사결정의 **비보상형(사전식) 규칙**에 따른 것이다. 거래 금지 의무는 다른 위험 요인이 낮다고 상쇄될 수 없기 때문이다.

### 2.5 치명도 정량화 (AHP)

카테고리 간 차이를 수치로 보이기 위해, 제재가 금융기관에 피해를 주는 경로를 네 가지로 나누고 **AHP(계층분석법)** 로 경로별 가중치를 산출했다. 쌍대비교 판단은 한국 금융기관을 가정한 분석자의 판단이다.

| 비교 | 판단 (Saaty 척도) | 근거 |
|---|---|---|
| 법적 금지 vs 처벌 | 2 | 금지 의무 위반이 처벌의 원인이므로 금지를 더 근본적으로 봄 |
| 법적 금지 vs 환거래 상실 | 3 | 환거래 상실은 거래 상대 은행의 판단에 따른 간접 영향 |
| 법적 금지 vs 평판 | 5 | 평판은 회복 가능하고 측정이 불확실함 |
| 처벌 vs 환거래 상실 | 2 | 제재금은 즉각적이고 확정적인 재무 손실 |
| 처벌 vs 평판 | 4 | 확정 손실과 불확실 손실의 차이 |
| 환거래 상실 vs 평판 | 3 | 대리은행 관계 단절은 해외 송금 업무 자체를 막음 |

이 판단으로 계산한 가중치는 법적 금지 0.47, 처벌 0.28, 환거래 0.17, 평판 0.07이며, **일관성 비율(CR)은 0.019**로 기준(0.1 미만)을 충족했다. 카테고리별 경로 점수는 분석자가 직접 매긴 값이다.

### 2.6 뉴스 수집: GDELT에서 구글 뉴스 RSS로 전환

처음에는 알바니아와 같이 GDELT 뉴스 API를 사용했다. 알바니아에서 "요청 실패"와 "기사 0건"이 구분되지 않아 조용한 실패가 발생했던 점을 반영해, 본 수집 전에 현직 대통령 이름으로 **API 정상 여부를 먼저 테스트**하고 실패하면 중단하도록 안전장치를 넣었다.

실행 결과 이 안전장치가 작동했다. GDELT가 **429(요청 과다)** 를 반환했고, 30초·60초·120초로 대기 시간을 늘리는 지수 백오프에도 해제되지 않았다. 제한을 우회하는 방법은 수집 윤리에 어긋나므로 사용하지 않고, 대신 **설계 자체를 바꿨다.**

| 항목 | 기존 설계 | 변경 설계 |
|---|---|---|
| 검색 대상 | 의원 전원 (최대 560회 요청) | 제재 관련 인물만 (2회 요청) |
| 뉴스 소스 | GDELT API | 구글 뉴스 RSS (프랑스어·콩고판) |
| 검색어 | 공식 명단 이름 그대로 | "이름 성" / "성 이름" 두 어순 OR 검색 |

인물별 최신 헤드라인 5건은 기계 번역으로 한글 번역을 붙였으며, 번역 엔진 접속 실패에 대비해 엔진 자동 전환과 오류 표시를 구성했다.

## 3. 수행 과정과 결과 (EDA)

### 3.1 명단 발췌와 스크리닝 개요

`/pep/` 페이지에서 DR Congo 관련 데이터셋은 1개(**DR Congo Members of the National Assembly**, `cd_national_assembly`)였다. 상태 코드 200으로 정상 수신했고, CSV에서 **인물 560명**을 확보했다. 표의 엔티티 수(561)와 1건 차이는 인물이 아닌 항목이나 업데이트 시점 차이로 보이며 영향은 작다고 판단했다.

**표 1. 스크리닝 개요**

| 항목 | 값 |
|---|---|
| 의원 수 | 560명 |
| 대조한 제재명단 | 73,810건 |
| 제재 확정 (ID 일치) | 1명 |
| 동일인 가능성 높은 후보 (이름 + 생년월일 일치) | 1명 |
| 오탐 (생년월일 불일치) | 1명 |
| 실제 제재 대상 비율 | 2 / 560 = 0.36% |

### 3.2 질문 1: 제재 대상인가

| 판정 | 인물 | 근거 | 제재 목록 수 |
|---|---|---|---|
| 확정 | Bitakwira Bihona Hayi Justin | OpenSanctions ID 일치 | 7 |
| 동일인 가능성 높음 | Ndjabu Ngabu Floribert | 이름(Floribert Ngabu) + 생년월일 일치 | 15 |
| 오탐 | Wagner Tshienda Tshienda Richard | 이름 단어만 겹침(Richard Wagner), 생년월일 불일치 | 1 |

**Bitakwira Bihona Hayi Justin**은 OpenSanctions가 의원 명단과 제재명단에서 동일 인물로 연결해 둔 사례로, 가장 신뢰도가 높은 결과다.

**Ndjabu Ngabu Floribert**는 OpenSanctions가 아직 ID로 연결하지 않아 1단계에서는 잡히지 않았다. 그러나 개선된 2단계 매칭에서 제재명단의 "Floribert Ngabu"가 공식 이름에 포함된 것으로 확인됐고, 3단계에서 **생년월일까지 일치**했다. 알바니아에서 쓴 정확 일치 방식이었다면 놓쳤을 사례로, 매칭 방식 개선의 효과를 직접 보여준다.

**Wagner Tshienda Tshienda Richard**는 "Richard"와 "Wagner"가 이란 제재명단의 다른 인물 "Richard Wagner"와 겹쳐 후보로 잡혔으나, 생년월일이 일치하지 않아 오탐으로 판정했다. 넓게 잡는 매칭의 위험이 실제로 나타났고, 생년월일 대조가 이를 정확히 걸러냈다. **이름만으로 판단했다면 무고한 의원이 이란 제재 대상으로 오인됐을 것이다.**

### 3.3 제재 출처와 카테고리 분포

오탐을 제외한 두 인물의 제재 목록 22개를 라벨링한 결과는 다음과 같다.

**표 2. 목록별 라벨링**

| 인물 | 제재 목록 | 원지정 기관 | 구분 | 조치 | 카테고리 |
|---|---|---|---|---|---|
| Ndjabu Ngabu Floribert | UN Security Council Consolidated Sanctions | UN 안보리 | 원지정 | 자산동결·여행금지 | A |
| Ndjabu Ngabu Floribert | Australian Sanctions Consolidated List | UN 안보리 | 이행(호주) | 자산동결 | A |
| Ndjabu Ngabu Floribert | Japan Economic Sanctions and List of Eligible People | UN 안보리 | 이행(일본) | 자산동결 | A |
| Ndjabu Ngabu Floribert | South Africa Targeted Financial Sanctions | UN 안보리 | 이행(남아공) | 자산동결 | A |
| Ndjabu Ngabu Floribert | Argentina RePET Sanctions | UN 안보리 | 이행(아르헨티나) | 자산동결 | A |
| Ndjabu Ngabu Floribert | EU Financial Sanctions Files (FSF) | EU | 원지정 | 자산동결 | B |
| Ndjabu Ngabu Floribert | Belgian Financial Sanctions | EU | 이행(벨기에) | 자산동결 | B |
| Ndjabu Ngabu Floribert | French National Asset Freezing System | EU | 이행(프랑스) | 자산동결 | B |
| Ndjabu Ngabu Floribert | Monaco National Fund Freezing List | EU | 이행(모나코) | 자산동결 | B |
| Ndjabu Ngabu Floribert | Swiss SECO Sanctions/Embargoes | EU | 이행(스위스) | 자산동결 | B |
| Ndjabu Ngabu Floribert | UK FCDO Sanctions List | 영국 | 원지정 | 자산동결 | B |
| Ndjabu Ngabu Floribert | Taiwan Strategic High-Tech Commodities Entity List | 대만 | 독자 | 수출통제 | B |
| Ndjabu Ngabu Floribert | EU Consolidated Travel Bans | EU | 원지정 | 여행금지 | C |
| Ndjabu Ngabu Floribert | EU Council Official Journal Sanctioned Entities | EU | 원지정 | 법적 지정 근거(관보) | C |
| Ndjabu Ngabu Floribert | Ukraine SFMS Blacklist | 우크라이나 | 독자 | FIU 감시목록 | C |
| Bitakwira Bihona Hayi Justin | EU Financial Sanctions Files (FSF) | EU | 원지정 | 자산동결 | B |
| Bitakwira Bihona Hayi Justin | Belgian Financial Sanctions | EU | 이행(벨기에) | 자산동결 | B |
| Bitakwira Bihona Hayi Justin | French National Asset Freezing System | EU | 이행(프랑스) | 자산동결 | B |
| Bitakwira Bihona Hayi Justin | Monaco National Fund Freezing List | EU | 이행(모나코) | 자산동결 | B |
| Bitakwira Bihona Hayi Justin | Swiss SECO Sanctions/Embargoes | EU | 이행(스위스) | 자산동결 | B |
| Bitakwira Bihona Hayi Justin | EU Consolidated Travel Bans | EU | 원지정 | 여행금지 | C |
| Bitakwira Bihona Hayi Justin | EU Council Official Journal Sanctioned Entities | EU | 원지정 | 법적 지정 근거(관보) | C |

**표 3. 인물별 카테고리 분포**

| 인물 | A (금지) | B (제한) | C (감시) | 목록 합계 |
|---|---|---|---|---|
| Ndjabu Ngabu Floribert | 5 | 7 | 3 | 15 |
| Bitakwira Bihona Hayi Justin | 0 | 5 | 2 | 7 |

두 인물의 제재 성격은 뚜렷하게 다르다. **Ndjabu**는 UN 안보리 지정과 그 이행 목록 5개가 A급을 이루고, EU·영국 지정이 B급으로 더해진다. 반면 **Bitakwira**는 A급이 하나도 없고, EU 자산동결과 그 회원국 이행이 B급, EU 여행금지와 관보 공시가 C급이다. 즉 Bitakwira는 **유럽 지역에 한정된 제재**, Ndjabu는 **UN을 기반으로 전 세계로 확산된 제재**다.

**표 4. 원지정 기관 기준 재집계 (중복 이행 묶음)**

| 인물 | 원지정 기관 | 해당 목록 수 | 그중 원지정 | 그중 타국 이행 |
|---|---|---|---|---|
| Ndjabu Ngabu Floribert | UN 안보리 | 5 | 1 | 4 |
| Ndjabu Ngabu Floribert | EU | 7 | 3 | 4 |
| Ndjabu Ngabu Floribert | 영국 | 1 | 1 | 0 |
| Ndjabu Ngabu Floribert | 대만 (수출통제) | 1 | 1 | 0 |
| Ndjabu Ngabu Floribert | 우크라이나 (FIU 목록) | 1 | 1 | 0 |
| Bitakwira Bihona Hayi Justin | EU | 7 | 3 | 4 |

표 3과 표 4를 비교하면 **목록 수와 실제 위험이 일치하지 않는다**는 점이 드러난다. Ndjabu의 15개 목록 중 8개는 UN·EU 결정의 타국 이행이며, 금융 제재로서 독립적인 결정은 UN, EU, 영국 세 곳이다. Bitakwira의 7개 목록은 사실상 EU의 결정 하나에서 나왔다. 단순히 목록 수로 비교하면 Ndjabu가 Bitakwira보다 두 배 이상 위험해 보이지만, 그 차이의 본질은 "목록이 많아서"가 아니라 **"UN이냐 EU냐"** 에 있다. 또한 대만 수출통제나 우크라이나 FIU 목록은 금융거래 금지가 아니므로 위험 판단에 미치는 영향이 작다.

### 3.4 카테고리별 치명도와 최종 판정

**표 5. 카테고리별 치명도 (AHP)**

| 영향 경로 | AHP 가중치 | A | B | C |
|---|---|---|---|---|
| 법적 거래금지·동결 의무 | 0.47 | 1.0 | 0.3 | 0.0 |
| 제재금·형사처벌 | 0.28 | 1.0 | 0.5 | 0.1 |
| 환거래·달러 접근 상실 | 0.17 | 1.0 | 0.6 | 0.2 |
| 평판 | 0.07 | 1.0 | 0.7 | 0.5 |
| **치명도 지수** | CR = 0.019 | **1.00** | **0.44** | **0.10** |

A급을 1로 볼 때 B급은 약 0.44, C급은 약 0.10이다. B급이 구속력이 없는데도 A급의 절반 가까이 되는 이유는, 해외 대리은행들이 EU·영국 제재명단으로 스크리닝하기 때문에 **송금이 중간에서 막히거나 거래 관계가 끊길 위험**(환거래 0.6)이 상당하고, 거래가 나중에 EU나 달러와 연결되면 처벌 위험이 생기기 때문이다. 반면 C급은 금융거래를 막지 않아 대부분 평판 위험에 그친다. 막연히 "강한 제재 / 약한 제재"로 나눌 때보다 차이가 훨씬 크게 나타나는데, 금융거래에 실제로 치명적인 것은 **법적 금지 의무와 처벌**이기 때문이다.

**표 6. 인물별 최종 판정**

| 인물 | 판정 | 최종 카테고리 | 가장 강한 제재 | 독립 원지정 기관 | 치명도 | 처리 방식 | 수집 기사 |
|---|---|---|---|---|---|---|---|
| Ndjabu Ngabu Floribert | 동일인 후보 | A | UN 안보리 · 자산동결 | UN 안보리, EU, 영국 | 1.00 (오버라이드) | 거래 금지·동결·보고 | 9건 |
| Bitakwira Bihona Hayi Justin | 확정 | B | EU · 자산동결 | EU | 0.44 | EDD·상급자 승인·고위험 하한 | 74건 |
| Wagner Tshienda Tshienda Richard | 오탐 | 해당 없음 | 해당 없음 | 해당 없음 | 0 | 스크리닝 대상에서 제외 | 0건 |

판정의 신뢰도(확정 vs 후보)와 위험의 크기(A vs B)가 서로 반대라는 점이 흥미롭다. **Bitakwira는 동일인 여부가 확정됐지만 위험은 B급**이고, **Ndjabu는 동일인 여부가 후보 단계지만 위험은 A급**이다. 실무적으로는 A급 후보인 Ndjabu의 동일인 확인을 최우선으로 처리해야 한다. A급은 오판 시 대가가 가장 크기 때문이다.

### 3.5 기관 가정에 따른 민감도

**표 7. 기관 가정별 카테고리**

| 기관 가정 | Ndjabu Ngabu Floribert | Bitakwira Bihona Hayi Justin |
|---|---|---|
| 기본 (EU·영국 연결 없음, 달러 결제 있음) | A | B |
| EU 연결 있음 (`eu_nexus = True`) | A | A |
| 영국 연결 있음 (`uk_nexus = True`) | A | B |
| 달러 결제 없음 (`usd_nexus = False`) | A | B |

Ndjabu는 어떤 가정에서도 A급이다. UN 제재는 회원국 전체에 구속력이 있기 때문이다. 반면 Bitakwira는 **기관에 EU 연결고리가 있으면 B급에서 A급으로 올라간다.** 유럽에 지점이 있는 은행이라면 Bitakwira와의 거래는 위험 관리의 문제가 아니라 법적 금지의 문제가 된다. 제재 위험은 사람의 속성만으로 정해지는 것이 아니라 **사람과 기관의 관계**로 정해진다는 점을 보여준다.

### 3.6 질문 2: 뉴스 헤드라인을 찾을 수 있는가

2.6절의 설계 변경 후 구글 뉴스 RSS로 수집했으나, 첫 실행에서 두 인물 모두 **응답 302, 기사 0건**이 나왔다. 알바니아의 경험에 따라 0건을 그대로 결론으로 받아들이지 않고 원인을 확인했다. 첫째, 302는 리다이렉트 응답으로 오류가 아니었는데 코드가 200이 아닌 응답을 모두 버리고 있었다. 리다이렉트를 따라가도록 수정했다. 둘째, 공식 명단은 성이 앞에 오지만 기사는 "Justin Bitakwira"처럼 이름을 앞에 쓴다. 두 어순을 모두 검색하도록 바꿨다.

| 인물 | 수집 기사 수 |
|---|---|
| Bitakwira Bihona Hayi Justin | 74건 |
| Ndjabu Ngabu Floribert | 9건 |
| 합계 | 83건 |

두 인물 모두 기사를 확보했으며, 인물별 최신 헤드라인 5건은 원문과 한글 번역으로 정리해 `drc_news_top5_ko.csv`에 저장했다.

언론 노출은 **B급인 Bitakwira(74건)에 집중**되어 있고, A급인 Ndjabu는 9건에 그친다. 기사 수가 위험의 크기와 반대로 나타난 것이다. Bitakwira는 현직 정치인으로서의 의정 활동 보도가 많을 수 있어, 기사 수는 위험이 아니라 **공인으로서의 노출량**에 가깝다. 따라서 기사는 위험 점수를 그대로 올리는 근거가 아니라, 제재 관련 내용인지 분류하고 상태 변화(추가 지정, 해제, 회피 정황)를 감지하는 **동향 지표**로 써야 한다. 특히 Bitakwira처럼 구속력 없는 B급 제재 대상은 법적 지위만으로 결론이 나지 않으므로, 기사 동향이 위험 판단에서 더 큰 역할을 한다.

## 4. 알바니아와의 비교

| 항목 | 알바니아 | DR Congo |
|---|---|---|
| 의원 수 | 140명 | 560명 |
| 제재 확정 | 0명 | 1명 |
| 동일인 가능성 높은 후보 | 0명 | 1명 |
| 오탐 | 해당 없음 | 1명 |
| 뉴스 확보 | 실패 (이름 형식·429) | 성공 (83건) |
| 이름 매칭 | 정확 일치 | 어순 무관 단어 묶음 + 생년월일 |
| 뉴스 검색 구조 | 전원 개별 검색 | 제재 인물만 검색 |
| 제재 분석 깊이 | 해당 없음 | 출처 라벨링, A/B/C 카테고리, AHP 치명도 |

같은 수집 파이프라인이라도 **매칭 방식과 검색 구조를 개선하자 결과가 크게 달라졌다.** 특히 Ndjabu Ngabu Floribert는 알바니아에서 쓴 정확 일치 방식이었다면 찾지 못했을 인물이며, 그가 A급이라는 점에서 놓쳤을 때의 대가도 가장 컸을 것이다. 따라서 알바니아의 "제재 0명"도 실제로 대상이 없는 것인지, 매칭 방식의 한계였는지는 개선된 방식으로 다시 확인할 필요가 있다.

## 5. 목표 질문에 대한 답

**질문 1 (제재 대상인가).** DR Congo 국회의원 560명 중 **2명(0.36%)이 제재 대상**이다. **Ndjabu Ngabu Floribert**는 이름과 생년월일이 일치해 동일인일 가능성이 높으며, UN 안보리 자산동결을 기반으로 한 **A급(거래 금지)** 이다. **Bitakwira Bihona Hayi Justin**은 ID 일치로 확정되었으며, EU 자산동결 중심의 **B급(EDD·거래 제한)** 이고, 기관이 EU와 연결되어 있다면 A급이 된다. 이름만 겹친 1명은 생년월일 불일치로 오탐 처리했다.

**질문 2 (뉴스 헤드라인을 찾을 수 있는가).** **찾을 수 있다.** 제재 인물 2명에 대해 구글 뉴스 RSS로 총 83건의 기사를 확보했고, 인물별 최신 헤드라인을 원문과 한글 번역으로 정리했다. 다만 기사 수는 위험의 크기와 일치하지 않았으며, 기사는 위험 점수보다 동향 감지 지표로 활용하는 것이 적절하다.

## 6. 실무 KYC 관점에서의 의미

이번 분석은 KYC 스크리닝의 핵심 흐름인 "명단 데이터 → 이름 대조 → 오탐 해소 → 부정적 언론 확인 → 지속 모니터링"을 재현했다. 특히 실무에서 가장 많은 인력이 드는 **오탐 해소**를 생년월일 대조로 구현했고, 제재 일치를 **구속력과 조치 종류에 따라 등급화**해 "거래 금지"와 "강화된 확인"을 구분했다.

다만 실무와 다른 점도 있다. 실무는 국가 단위 명단이 아니라 **기관의 고객**에서 출발하고, 얼럿마다 담당자가 판단 근거를 기록하며, 상용 워치리스트와 본문 기반 언론 분석 도구를 사용한다. 또 PEP이면서 제재 대상인 경우 실무에서는 제재가 우선하므로, A급은 EDD가 아니라 거래 차단과 당국 보고로 이어진다.

## 7. 한계

"동일인 가능성 높음"은 이름과 생년월일에 근거한 판단이므로, 최종 확정은 원 제재 목록의 상세 정보(국적, 신분증 번호, 지정 사유 등)로 해야 한다. 제재 명목(지정 사유)은 OpenSanctions 데이터만으로 충분히 확인되지 않아 원 공시(UN 안보리, EU 관보) 확인이 필요하다. 목록별 원지정 기관과 조치 종류는 목록 이름을 기준으로 분류했으므로 실제 조치 내용과 다를 수 있다.

카테고리 기준, 기관 가정, AHP 쌍대비교 판단과 카테고리별 경로 점수는 모두 **설명을 위한 분석자의 가정**이다. 실무에 적용하려면 컴플라이언스·법무 전문가 다수의 판단을 기하평균으로 통합하고, 판단을 바꿔도 결론이 유지되는지 민감도 분석으로 검증해야 한다.

이번 대조는 "제재(sanctions)" 데이터만 포함했으므로 입국 금지나 부패 관련 지정처럼 다른 분류의 위험 정보는 범위 밖이다. 구글 뉴스 RSS는 과거 기사도 함께 반환하므로 기사 수가 곧 최근 노출량을 뜻하지 않으며, 헤드라인 번역은 기계 번역이다. 또한 GDELT 제한으로 뉴스 수집을 제재 인물 2명에게만 적용했는데, 실무적으로 뉴스의 가치는 오히려 **제재 대상이 아닌 나머지 558명의 조기 위험 탐지**에 있다. PEP 분류, 제재 여부, 기사 수는 그 자체로 개인의 위법을 단정하지 않는다.

## 8. 윤리 점검

OpenSanctions는 robots.txt를 확인해 허용 경로(`/pep/`, `/datasets/`)만 접근했고, 금지 경로(`/search`, `/entities`)는 공식 다운로드 파일로 대체했으며, 요청 간격 10초(Crawl-delay)를 지켰다. GDELT가 429를 반환했을 때는 재시도 간격을 늘린 뒤 수집을 중단했고, IP 변경 같은 우회 방법은 쓰지 않았다. 뉴스는 자동 수신용 공개 형식인 RSS로 받고 인물별 요청 사이 3초 간격을 두었으며, 저작권을 고려해 기사 본문은 저장하지 않았다. 데이터는 비상업 학습용(CC BY-NC)으로만 사용하며, 명단과 기사 CSV는 `.gitignore`로 저장소에서 제외해 재배포하지 않았다.

## 9. 배운 점

첫째, **매칭 방식이 결과를 결정한다.** 알바니아와 DR Congo의 차이는 국가의 차이이기도 하지만, 그보다 이름을 어떻게 비교했느냐의 차이가 컸다. 공식 명단의 이름 형식(부칭, 후성, 어순)을 이해하지 못하면 수집 도구가 아무리 정확해도 매칭은 실패한다.

둘째, **넓게 잡으면 반드시 걸러야 한다.** 어순 무관 매칭으로 놓쳤던 A급 제재 인물을 찾았지만, 같은 방식이 무고한 의원을 이란 제재 대상 후보로 올리기도 했다. 탐지 범위를 넓히는 것과 생년월일 같은 두 번째 식별자로 검증하는 것은 반드시 함께 가야 한다.

셋째, **"제재 대상"은 하나의 범주가 아니다.** 같은 제재 대상이라도 원지정 기관과 조치 종류에 따라 거래 금지부터 모니터링까지 처리가 완전히 달라지고, 목록 수는 위험의 크기를 그대로 반영하지 않는다. 또한 같은 사람이라도 기관이 어디와 연결되어 있느냐에 따라 등급이 바뀐다.

넷째, **0건과 에러는 의심하고, 제한은 우회하지 않고 설계로 해결한다.** GDELT 테스트 안전장치, RSS 302 응답, 번역 실패 모두 "결과가 없다"로 보였지만 실제로는 각기 다른 원인이 있었다. 특히 429 제한에 부딪혔을 때 요청 대상을 560명에서 2명으로 좁히는 설계 변경으로 해결한 것은, 책임 있는 수집이 효율적인 수집이기도 하다는 점을 보여줬다.